In [ ]:
import numpy as np
from collections import defaultdict
import math as mt

import sys
import os
import time as timer
from tqdm import tqdm
import glob

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.ticker import LogLocator
import matplotlib.colors as cls
from mpl_toolkits.mplot3d import Axes3D

markers = ['+','x','o','v','^','<','>','s','p','*','h','H','D','d','|','_','1','2','3','4','.', ',',]
linestyles = ['-', '--', '-.', ':', '']

folder_path = "../data/"

mod_name = ["precede_adder",
            "product_adder",
            "full_adder",
            "chain",
            "cube",
            "mixing",
            "xor"]

no_phases_mod = ["precede_adder",
                 "cube",
]

def gate_stage_num(N,name):
    if name == "product_adder":
        return N+1
    elif name == "precede_adder":
        return N+2
    elif name == "full_adder":
        return 2*N+1
    elif name == "straight":
        return 2*N+1
    else:
        return N

def read_data_set(name,mod,rate):
    data_dic = {}
    files = glob.glob(folder_path+name+"/"+name+"*mod-"+str(mod)+"_*rate?"+str(rate)+"-*.csv")
    for file in files:
        data_dic[file] = pd.read_csv(file,header=None).to_numpy()
    data_list = np.empty((0, data_dic[files[0]][0].shape[0]))
    for file in files:
        data_list = np.vstack((data_list,pd.read_csv(file,header=None).to_numpy()))
    return data_list


In [ ]:
def bootstrap(data):
    n_iterations = 500       # ブートストラップの反復回数
    n_size = len(data)         # 各リサンプリングのサンプルサイズ

    bootstrap_means = []
    for _ in range(n_iterations):
        # サンプルからリサンプリング（復元抽出）
        sample = np.random.choice(data, size=n_size, replace=True)
        # リサンプルしたデータの平均を計算
        bootstrap_means.append(np.mean(sample))
    conf = np.std(bootstrap_means)
    stderr = conf*mt.sqrt(n_size)
    return np.mean(bootstrap_means),conf,stderr

def name_to_job(file):
    job = os.path.basename(file).split("_with")[0]
    job = job.split("test")[0]+"rate"+job.split("rate")[1]
    name = job.split(":")[0]
    mod = int(job.split(":mod-")[1].split("_rate")[0])
    rate = float(job.split("rate:")[1].split("-")[0])
    return (name,mod,rate)

def time_data_picking(file):
    job = name_to_job(file)
    temporary_list = pd.read_csv(file,header=None,usecols=[2]).to_numpy()
    return job,temporary_list

def confidence(name):
    files = glob.glob(folder_path+name+"/"+name+"*test*rate*:*_with*.csv")
    data_dic = defaultdict(lambda:np.array([]))
    temporary_list = []
    parameter_list = []
    mod=0
    rate=0
    time=0
    conf=0
    case=0
    for file in files:
        job,temporary_list = time_data_picking(file)
        data_dic[job] = np.append(data_dic[job],temporary_list)
    for key in data_dic:
        if not key[0] == name : print("?")
        mod = key[1]
        rate = key[2]
        time = np.mean(data_dic[key])
        _,conf,stderr = bootstrap(data_dic[key])
        case = len(data_dic[key])
        parameter_list.append({"name":name,"mod":int(mod),"rate":float(rate),"time":time,"mean_stderr":conf,"trial":case,"std":stderr})
    parameter_list = sorted(parameter_list,key=lambda x: (x["mod"],x["rate"]))
    data_frame = pd.DataFrame(parameter_list)
    data_frame.to_csv("./data-"+name+".csv",index=False)
    print("file: "+name+".csv is outputed")

# for name in mod_name:
#    confidence(name)


In [ ]:
def mixing_csv():
    #データの取得
    name = "mixing"
    files = glob.glob(folder_path+name+"/"+name+"*test*rate*:*_with*.csv")
    joblist = sorted(set([((os.path.basename(file).split("mod-")[1].split("_test")[0]),(os.path.basename(file).split("rate:")[1].split("-")[0])) for file in files]))
    
    for carry in [",C",",not C"]:
        data_dic = defaultdict(lambda:np.array([]))
        parameter_list = []
        time=0
        stderr=0
        case=0
        for (mod,rate) in joblist:
            data_frame = pd.DataFrame()
            files = glob.glob(folder_path+name+"/"+name+":mod-"+mod+"_*rate:"+rate+"-*_with*.csv")
            for file in files:
                data_dic[file] = pd.read_csv(file,header=None).to_numpy()
            data_list = np.empty((0, data_dic[files[0]][0].shape[0]))
            for file in files:
                data_list = np.vstack((data_list,data_dic[file]))
            data_frame = pd.DataFrame(data_list)
            data_frame.columns = ["input","output","time","jump"]
            data_frame = data_frame.sort_values("input")
            data_frame = data_frame[data_frame["input"].str.contains(carry)]
            time,conf,_ = bootstrap(data_frame["time"])
            stderr = np.std(data_frame["time"])
            case = len(data_frame["time"])
            parameter_list.append({"name":name,"mod":int(mod),"rate":float(rate),"time":time,"mean_stderr":conf,"trial":case,"std":stderr})
        parameter_list = sorted(parameter_list,key=lambda x: (x["mod"],x["rate"]))
        #print(parameter_list)
        data_frame = pd.DataFrame(parameter_list)
        if carry == ",C":
            data_frame.to_csv("./data-"+name+"-C.csv",index=False)
        elif carry == ",not C":
            data_frame.to_csv("./data-"+name+"-NC.csv",index=False)
    print("file: "+name+".csv is outputed")

# mixing_csv()